# Download Data From Kaggle

For more information about the competition check the [link](https://www.kaggle.com/competitions/titanic/)

In [1]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("Running on Google Colab. Setting up Kaggle data...")
    from google.colab import userdata
    import json

    # Setup Kaggle API
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

    !mkdir -p ~/.kaggle
    kaggle_config = {
        "username": os.environ["KAGGLE_USERNAME"],
        "key": os.environ["KAGGLE_KEY"],
    }
    with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as f:
        json.dump(kaggle_config, f)
    !chmod 600 ~/.kaggle/kaggle.json

    # Download data if it doesn't exist yet
    if not os.path.exists("data"):
        !kaggle competitions download -c titanic
        !unzip -q titanic.zip -d data/
        !rm titanic.zip
        print("Data downloaded and unzipped.")
else:
    if not os.path.exists("data"):
        print("Download data ...")
        !kaggle competitions download -c titanic
        !unzip -q titanic.zip -d data/
        !rm titanic.zip
        print("Data downloaded")
    else:
        print("Data already exists")

Data already exists


# Exploring The Dataset

In [2]:
import pandas as pd

df = pd.read_csv("data/train.csv")

In [3]:
mean_age = df["Age"].mean()
df["Age"] = df["Age"].fillna(mean_age)
df["Embarked"] = df["Embarked"].fillna("S")
df["Cabin"] = df["Cabin"].ffill()
df.drop(
    ["Embarked", "Cabin", "Ticket", "Name", "PassengerId", "Fare"], axis=1, inplace=True
)

In [7]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df["Sex"] = le.fit_transform(df["Sex"])

df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch
0,0,3,1,22.0,1,0
1,1,1,0,38.0,1,0
2,1,3,0,26.0,0,0
3,1,1,0,35.0,1,0
4,0,3,1,35.0,0,0


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import Normalizer

Y = df["Survived"]
X = df.drop(["Survived"], axis=1)

X = Normalizer().fit_transform(X)

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42
)

In [ ]:
from sklearn.svm import SVC


from sklearn.model_selection import RandomizedSearchCV, KFold
from sklearn.preprocessing import StandardScaler

from scipy.stats import loguniform


svm = SVC()

param_distribution = [
    {
        "C": loguniform(1e-5, 15),
        "kernel": ["rbf", "linear", "poly", "sigmoid"],
        "degree": [2, 3, 4, 5, 6, 7, 8, 9],
        "shrinking": [True, False],
        "probability": [True, False],
        "class_weight": ["balanced", None],
        "tol": loguniform(1e-5, 15),
        "cache_size": [200, 500, 700, 900, 1100, 1300, 1500],
        "max_iter": [10000],
        "break_ties": [True, False],
        "decision_function_shape": ["ovo", "ovr"],
        "coef0": loguniform(1e-5, 15),
    },
]

search = RandomizedSearchCV(
    estimator=svm,
    param_distributions=param_distribution,
    verbose=3,
    n_iter=100,
    cv=KFold(n_splits=10),
    n_jobs=-1,
)


search.fit(X_train, Y_train)
print(f"best_score: {search.best_score_}")
print(f"best_params: {search.best_params_}")

Fitting 10 folds for each of 100 candidates, totalling 1000 fits
best_score: 0.83141627543036
best_params: {'clf': SVC(), 'clf__C': np.float64(0.2824145204016417), 'clf__class_weight': None, 'clf__coef0': np.float64(1.2404957737885998), 'clf__degree': 8, 'clf__kernel': 'rbf', 'clf__tol': np.float64(0.8061955663737288), 'scaller': StandardScaler()}


In [278]:
pred = search.best_estimator_.predict(X_test)
scores = search.best_estimator_.score(X_test, Y_test)
scores

0.8156424581005587

In [275]:
!ls

data  sample_data  submission.csv


In [279]:
# df_test = pd.read_csv("data/test.csv")
# mean_age = df_test["Age"].mean()
# df_test["Age"] = df_test["Age"].fillna(mean_age)
# df_test["Embarked"] = df_test["Embarked"].fillna("S")
# df_test["Cabin"] = df_test["Cabin"].ffill()
# df_test.shape[0]
# df_test.drop(
#     ["Embarked", "Cabin", "Ticket", "Name", "Fare"], axis=1, inplace=True
# )
# le = LabelEncoder()
# df_test["Sex"] = le.fit_transform(df_test["Sex"])

# preds = search.best_estimator_.predict(df_test.drop("PassengerId", axis=1))
# preds

# output = pd.DataFrame({'PassengerId': df_test["PassengerId"], 'Survived': preds})
# output.to_csv('submission.csv', index=False)

# !kaggle competitions submit -c titanic -f submission.csv  -m "Message"


100% 2.77k/2.77k [00:00<00:00, 3.00kB/s]
Successfully submitted to Titanic - Machine Learning from Disaster